# Figure generation script

In [ ]:
"""
Generate Figure 6: Representative Power Spectral Density (PSD)
===============================================================
Plots the Welch PSD for the 600g trial 1 data, showing a single
dominant peak at the fundamental pendulum frequency. The absence
of secondary peaks confirms that the time-domain frequency bias
arises from Coulomb-induced waveform distortion, not a secondary
oscillatory mode.

Reference:
  "Systematic Time-Frequency Discrepancy as a Signature of Nonlinear
   Friction: Experimental Evidence from a Coulomb-Damped Oscillator"
   Muhammad et al., ICMASC 2026.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from QSignature import QSmooth
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import os

# Setup
OUTPUT_DIR = "./figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

# Load data (update path as needed)
df = pd.read_csv("path/to/data/600g t1.csv", index_col=0)
t = df.index.values
R_raw = df.iloc[:, 0].values

# Detrend
R_detrend = R_raw - np.mean(R_raw)

# Sampling frequency
dt = np.mean(np.diff(t))
fs = 1.0 / dt

# Welch PSD
nperseg = min(256, len(R_detrend) // 2)
freqs, psd = welch(R_detrend, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)

# Dominant peak
valid_idx = freqs > 0.05  # exclude DC
peak_idx = np.argmax(psd[valid_idx])
f0 = freqs[valid_idx][peak_idx]
psd_peak = psd[valid_idx][peak_idx]

# ---- FIGURE ----
fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogy(freqs, psd, 'b-', linewidth=1.5, label='PSD')
ax.scatter(
    f0, psd_peak, color='red', s=150,
    edgecolor='black', linewidth=1.5, zorder=10,
    label=f'$f_0 = {f0:.3f}$ Hz',
)
ax.axvline(x=f0, color='red', linestyle='--', linewidth=1.5, alpha=0.6)
ax.axvspan(0, 0.05, alpha=0.1, color='gray',
           label='DC exclusion ($f < 0.05$ Hz)')

ax.annotate(
    f'$f_0 = {f0:.3f}$ Hz\n(Peak PSD = {psd_peak:.2e} m$^2$/Hz)',
    xy=(f0, psd_peak),
    xytext=(f0 + 0.05, psd_peak * 1.5),
    arrowprops=dict(arrowstyle='->', color='gray', lw=1),
    fontsize=11,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
)

ax.set_xlabel('Frequency (Hz)', fontsize=14)
ax.set_ylabel('Power Spectral Density (m$^2$/Hz)', fontsize=14)
ax.set_title('Figure 6: Representative Power Spectral Density (600g Trial 1)',
             fontsize=16)
ax.set_xlim([0, 1.0])
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='upper right', fontsize=10)

# Inset: smoothed time series
inset_ax = inset_axes(ax, width="30%", height="25%", loc='upper left')
qs = QSmooth()
R_smooth = qs.savgol(t, R_raw, window_frac=0.05, polyorder=3)
inset_ax.plot(t, R_smooth, 'k-', linewidth=1)
inset_ax.set_xlabel('Time (s)', fontsize=8)
inset_ax.set_ylabel('Displacement (m)', fontsize=8)
inset_ax.set_title('Smoothed $R(t)$', fontsize=9)
inset_ax.grid(True, alpha=0.3)
inset_ax.set_xlim([0, 20])
inset_ax.tick_params(axis='both', which='major', labelsize=7)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig6_psd_example.png'), dpi=300)
plt.savefig(os.path.join(OUTPUT_DIR, 'fig6_psd_example.pdf'))
plt.show()

print(f"Figure 6 saved to: {OUTPUT_DIR}")
print(f"Fundamental frequency: f₀ = {f0:.4f} Hz")
print(f"Peak PSD: {psd_peak:.4e} m²/Hz")
print("Note: Only a single dominant peak is observed, confirming that")
print("the time-frequency discrepancy arises from waveform distortion,")
print("not a secondary oscillatory mode.")

# Frequency analysis for data

In [ ]:
"""
Time-Frequency Discrepancy Diagnostic (delta_f) Computation
===========================================================
Computes the systematic discrepancy between time-domain (peak-to-peak) 
and spectral (Welch PSD) frequency estimates for a Coulomb-damped 
curved-track oscillator.

The diagnostic delta_f quantifies waveform distortion induced by
nonlinear friction:
  - delta_f ≈ 0  → sinusoidal waveform (linear viscous damping)
  - delta_f > 0  → waveform "triangularization" (Coulomb friction)

Reference:
  "Systematic Time-Frequency Discrepancy as a Signature of Nonlinear
   Friction: Experimental Evidence from a Coulomb-Damped Oscillator"
   Muhammad et al., ICMASC 2026.

Data: TimingPro Dynamics Car on curved metal track
      Five mass loadings (0g, 150g, 300g, 450g, 600g)
      Three replicate trials per mass (N = 15 total runs)

Requirements:
  pip install QSignature scipy pandas numpy
"""

import numpy as np
import pandas as pd
from scipy.signal import welch, find_peaks
from QSignature import QSmooth

qs = QSmooth()

# ============================================================================
# LOAD DATA
# ============================================================================
# NOTE: Update file paths to match your local directory structure.
data_dict = {
    "0g1": pd.read_csv("path/to/data/0g 500 t1.csv", index_col=0),
    "0g2": pd.read_csv("path/to/data/0g 500 t2.csv", index_col=0),
    "0g3": pd.read_csv("path/to/data/0g 500 t3.csv", index_col=0),
    "150g1": pd.read_csv("path/to/data/150g t1.csv", index_col=0),
    "150g2": pd.read_csv("path/to/data/150g t2.csv", index_col=0),
    "150g3": pd.read_csv("path/to/data/150g t3.csv", index_col=0),
    "300g1": pd.read_csv("path/to/data/300g t1.csv", index_col=0),
    "300g2": pd.read_csv("path/to/data/300g t2.csv", index_col=0),
    "300g3": pd.read_csv("path/to/data/300g t3.csv", index_col=0),
    "450g1": pd.read_csv("path/to/data/450g t1.csv", index_col=0),
    "450g2": pd.read_csv("path/to/data/450 t2.csv", index_col=0),
    "450g3": pd.read_csv("path/to/data/450 t3.csv", index_col=0),
    "600g1": pd.read_csv("path/to/data/600g t1.csv", index_col=0),
    "600g2": pd.read_csv("path/to/data/600g t2.csv", index_col=0),
    "600g3": pd.read_csv("path/to/data/600g t3.csv", index_col=0),
}

mass_groups = {
    "0g":   ["0g1", "0g2", "0g3"],
    "150g": ["150g1", "150g2", "150g3"],
    "300g": ["300g1", "300g2", "300g3"],
    "450g": ["450g1", "450g2", "450g3"],
    "600g": ["600g1", "600g2", "600g3"],
}

# ============================================================================
# COMPUTE TIME-DOMAIN AND SPECTRAL FREQUENCIES
# ============================================================================
print("=" * 90)
print("TIME-FREQUENCY DISCREPANCY DIAGNOSTIC (delta_f) COMPUTATION")
print("=" * 90)

time_results = {}

for name, df in data_dict.items():
    t = df.index.values
    R_raw = df.iloc[:, 0].values

    # Smooth displacement signal (Savitzky-Golay, 5% window, cubic)
    R_smooth = qs.savgol(t, R_raw, window_frac=0.05, polyorder=3)

    # ---- Time-domain: peak-to-peak frequency ----
    peaks, _ = find_peaks(R_smooth, distance=20)
    troughs, _ = find_peaks(-R_smooth, distance=20)

    if len(peaks) >= 2:
        peak_times = t[peaks]
        peak_periods = np.diff(peak_times)
        T_time = np.mean(peak_periods)
        f_time = 1.0 / T_time
        source = "peaks"
    elif len(troughs) >= 2:
        trough_times = t[troughs]
        trough_periods = np.diff(trough_times)
        T_time = np.mean(trough_periods)
        f_time = 1.0 / T_time
        source = "troughs"
    else:
        T_time = np.nan
        f_time = np.nan
        source = "none"

    # ---- Spectral: Welch PSD frequency ----
    dt = np.mean(np.diff(t))
    R_detrend = R_raw - np.mean(R_raw)
    fs = 1.0 / dt
    freqs, psd = welch(R_detrend, fs=fs, nperseg=min(256, len(R_detrend) // 2))
    valid_idx = freqs > 0.05  # Exclude DC component
    peak_idx = np.argmax(psd[valid_idx])
    f_spec = freqs[valid_idx][peak_idx]

    # ---- Diagnostic delta_f ----
    if not np.isnan(f_time):
        delta_f = (f_time - f_spec) / f_spec
        diff_pct = 100.0 * delta_f
        time_results[name] = {
            'T_time': T_time,
            'f_time': f_time,
            'f_spec': f_spec,
            'delta_f': delta_f,
            'diff_pct': diff_pct,
            'n_peaks': len(peaks),
            'n_troughs': len(troughs),
            'source': source,
        }
        print(f"{name}: f_time = {f_time:.4f} Hz, f_spec = {f_spec:.4f} Hz, "
              f"delta_f = {delta_f:.4f} ({diff_pct:+.2f}%) "
              f"[{source}, {len(peaks)} peaks, {len(troughs)} troughs]")
    else:
        time_results[name] = {
            'T_time': np.nan,
            'f_time': np.nan,
            'f_spec': f_spec,
            'delta_f': np.nan,
            'diff_pct': np.nan,
            'n_peaks': len(peaks),
            'n_troughs': len(troughs),
            'source': 'none',
        }
        print(f"{name}: Insufficient extrema, f_spec = {f_spec:.4f} Hz "
              f"({len(peaks)} peaks, {len(troughs)} troughs)")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================
print("\n" + "=" * 90)
print("SUMMARY BY MASS GROUP")
print("=" * 90)
print(f"{'Mass':<12} {'f_time (Hz)':<18} {'f_spec (Hz)':<18} {'delta_f':<12} {'Diff (%)':<12}")
print("-" * 90)

for mass, runs in mass_groups.items():
    f_time_vals = [
        time_results[r]['f_time']
        for r in runs
        if not np.isnan(time_results[r]['f_time'])
    ]
    f_spec_vals = [time_results[r]['f_spec'] for r in runs]
    delta_f_vals = [
        time_results[r]['delta_f']
        for r in runs
        if not np.isnan(time_results[r]['delta_f'])
    ]
    diff_vals = [
        time_results[r]['diff_pct']
        for r in runs
        if not np.isnan(time_results[r]['diff_pct'])
    ]

    if f_time_vals:
        print(
            f"{mass:<12} {np.mean(f_time_vals):.4f}±{np.std(f_time_vals):.4f}   "
            f"{np.mean(f_spec_vals):.4f}±{np.std(f_spec_vals):.4f}   "
            f"{np.mean(delta_f_vals):.4f}   {np.mean(diff_vals):+.2f}"
        )
    else:
        print(
            f"{mass:<12} {'nan':<18} "
            f"{np.mean(f_spec_vals):.4f}±{np.std(f_spec_vals):.4f}   "
            f"{'nan':<12} {'nan':<12}"
        )

# ============================================================================
# GRAND STATISTICS (PUBLISHED VALUES)
# ============================================================================
print("\n" + "=" * 90)
print("GRAND STATISTICS (N = 15 trials)")
print("=" * 90)

all_f_time = [
    time_results[r]['f_time']
    for r in data_dict
    if not np.isnan(time_results[r]['f_time'])
]
all_f_spec = [time_results[r]['f_spec'] for r in data_dict]
all_delta_f = [
    time_results[r]['delta_f']
    for r in data_dict
    if not np.isnan(time_results[r]['delta_f'])
]

print(f"f_time  = {np.mean(all_f_time):.4f} ± {np.std(all_f_time):.4f} Hz")
print(f"f_spec  = {np.mean(all_f_spec):.4f} ± {np.std(all_f_spec):.4f} Hz")
print(f"delta_f = {np.mean(all_delta_f):.4f} ± {np.std(all_delta_f):.4f}")
print()
print("Physical interpretation:")
print("  delta_f = 0    → sinusoidal waveform (linear viscous damping)")
print("  delta_f > 0    → waveform distortion (nonlinear/Coulomb friction)")
print(f"  Observed delta_f = {np.mean(all_delta_f):.3f} → 30% peak-timing bias "
      "due to Coulomb friction")